# 03 — ПИД-агент удержания полосы
> **v2 | Агент удержания полосы + ПИД-регулятор | ветка: `v2-agent-pid`**

> 🕐 Обновлён: 2026-04-10 00:20 МСК

**Время:** ~5 минут  
Демонстрируем работу агента: нейросеть (чёрный ящик) + ПИД-регулятор.  
Сравниваем: только нейросеть vs нейросеть + ПИД.  
Подбираем оптимальные коэффициенты Kp, Ki, Kd.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys, os

# Явно переходим в папку проекта
REPO_PATH = '/content/drive/MyDrive/diploma'
os.chdir(REPO_PATH)
sys.path.insert(0, REPO_PATH)

# Создаём нужные папки на Drive если их нет
os.makedirs('outputs', exist_ok=True)

import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from src.dataset_v2 import load_and_cache, THRESHOLD
from src.model_v2   import LaneCNN
from src.pid        import PIDController, tune_pid
from src.agent      import LaneKeepingAgent

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Устройство:', DEVICE)

In [ ]:
# Патч agent.py и pid.py — CTE по controls, integral_error по u(t)
import re

# --- agent.py: cte по controls ---
_path = '/content/drive/MyDrive/diploma/src/agent.py'
_txt  = open(_path).read()
_txt  = _txt.replace(
    'cte          = float(np.mean(np.abs(errors)))',
    'cte          = float(np.mean(np.abs(controls)))'
)
# run_episode_no_pid тоже
_txt  = _txt.replace(
    "cte=float(np.mean(np.abs(errors))),",
    "cte=float(np.mean(np.abs(controls))),"
)
open(_path, 'w').write(_txt)

# --- pid.py: integral_error по u ---
_path2 = '/content/drive/MyDrive/diploma/src/pid.py'
_txt2  = open(_path2).read()
_txt2  = _txt2.replace(
    'return float(np.mean([abs(h["e"]) for h in self._history]))',
    'return float(np.mean([abs(h["u"]) for h in self._history]))'
)
open(_path2, 'w').write(_txt2)

# Перезагружаем модули
import importlib, src.agent, src.pid
importlib.reload(src.pid)
importlib.reload(src.agent)
from src.agent import LaneKeepingAgent
from src.pid   import PIDController, tune_pid
print('Патч применён: CTE и grid search теперь считаются по управляющему сигналу u(t)')

In [ ]:
# Загружаем данные и модель (с авто-пересозданием кэша если нужно)
import glob, pathlib, pandas as pd

CACHE_PATH = '/content/data/cache.npy'
LOCAL_DATA  = '/content/data'

if not pathlib.Path(CACHE_PATH).exists():
    print("Кэш не найден — пересоздаю...")
    import zipfile

    ZIP_PATH = '/content/drive/MyDrive/diploma/data/dataset.zip'
    os.makedirs(LOCAL_DATA, exist_ok=True)

    img_dirs = glob.glob(f'{LOCAL_DATA}/**/IMG', recursive=True)
    if not img_dirs:
        print("Распаковываю zip...")
        with zipfile.ZipFile(ZIP_PATH, 'r') as z:
            z.extractall(LOCAL_DATA)

    csvs = glob.glob(f'{LOCAL_DATA}/**/*.csv', recursive=True)
    CSV_PATH = csvs[0]

    _df = pd.read_csv(CSV_PATH)
    for _col in _df.columns[:3]:
        if _df[_col].dtype == object and '\\' in str(_df[_col].iloc[0]):
            _df[_col] = _df[_col].apply(
                lambda x: 'IMG/' + str(x).strip().replace('\\', '/').split('/')[-1]
                if pd.notna(x) else x
            )
            _fixed = str(pathlib.Path(CSV_PATH).parent) + '/driving_log_fixed.csv'
            _df.to_csv(_fixed, index=False)
            CSV_PATH = _fixed
            break

    IMAGES_DIR = str(pathlib.Path(CSV_PATH).parent) + '/'
    images, angles = load_and_cache(CSV_PATH, IMAGES_DIR, CACHE_PATH)
else:
    images, angles = load_and_cache(None, None, CACHE_PATH)

# Берём последние 15% как тестовый трек
n_test = int(len(images) * 0.15)
test_frames = images[-n_test:]
test_angles = angles[-n_test:]
print(f'Тестовый трек: {n_test} кадров')

# Сначала пробуем локальную модель, если нет — берём с Drive
model_path = '/content/models/model_v2.pth'
if not os.path.exists(model_path):
    model_path = 'models/model_v2.pth'
    print('Локальной модели нет — загружаю с Drive')
agent = LaneKeepingAgent.load(model_path, device=DEVICE)

In [ ]:
# ── 1. Прогон без ПИД (только нейросеть) ─────────────────────────
result_raw = agent.run_episode_no_pid(test_frames)
print('БЕЗ ПИД:  ', result_raw.summary())

In [ ]:
# ── 2. Прогон с ПИД (базовые коэффициенты) ───────────────────────
result_pid = agent.run_episode(test_frames)
print('С ПИД:    ', result_pid.summary())

improvement = (1 - result_pid.cte / result_raw.cte) * 100
print(f'\nСнижение CTE: {improvement:.1f}%')

In [ ]:
# ── 3. Подбор оптимальных Kp, Ki, Kd ─────────────────────────────
# Используем предсказания нейросети на валидационном треке
n_val = int(len(images) * 0.15)
val_frames = images[int(len(images)*0.7): int(len(images)*0.85)]

# Получаем предсказания нейросети
agent.model.eval()
val_errors = np.array([agent.predict_error(f) for f in val_frames])

best = tune_pid(val_errors, steps=5)
print(f"\nОптимальные: Kp={best['Kp']:.3f}, Ki={best['Ki']:.4f}, Kd={best['Kd']:.3f}")

In [ ]:
# ── 4. Прогон с оптимальным ПИД ──────────────────────────────────
agent.pid = PIDController(Kp=best['Kp'], Ki=best['Ki'], Kd=best['Kd'])
result_opt = agent.run_episode(test_frames)
print('Оптимальный ПИД:', result_opt.summary())

In [ ]:
# ── 5. Визуализация: e(t), u(t), награды ─────────────────────────
T = min(200, len(test_frames))  # первые 200 кадров
t = np.arange(T)

fig = plt.figure(figsize=(14, 10))
gs  = gridspec.GridSpec(3, 1, hspace=0.4)

# График 1: ошибка e(t)
ax1 = fig.add_subplot(gs[0])
ax1.plot(t, result_raw.errors[:T], 'steelblue', alpha=0.7, label='Нейросеть (e)')
ax1.plot(t, test_angles[:T], 'gray', alpha=0.5, linestyle='--', label='Реальный угол')
ax1.axhline( THRESHOLD, color='red', linestyle=':', alpha=0.6)
ax1.axhline(-THRESHOLD, color='red', linestyle=':', alpha=0.6, label=f'±{THRESHOLD}')
ax1.fill_between(t, -THRESHOLD, THRESHOLD, alpha=0.08, color='green', label='Зона полосы')
ax1.set_ylabel('Ошибка e(t)')
ax1.set_title('Ошибка отклонения от полосы')
ax1.legend(fontsize=8)
ax1.grid(alpha=0.3)

# График 2: управляющий сигнал u(t)
ax2 = fig.add_subplot(gs[1])
ax2.plot(t, result_raw.controls[:T],  'steelblue', alpha=0.7, label='Без ПИД')
ax2.plot(t, result_opt.controls[:T],  'darkorange', linewidth=1.5, label='С ПИД (оптим.)')
ax2.axhline( THRESHOLD, color='red', linestyle=':', alpha=0.6)
ax2.axhline(-THRESHOLD, color='red', linestyle=':', alpha=0.6)
ax2.fill_between(t, -THRESHOLD, THRESHOLD, alpha=0.08, color='green')
ax2.set_ylabel('Управление u(t)')
ax2.set_title('Управляющий сигнал руля')
ax2.legend(fontsize=8)
ax2.grid(alpha=0.3)

# График 3: накопленная награда
ax3 = fig.add_subplot(gs[2])
ax3.plot(t, np.cumsum(result_raw.rewards[:T]),  'steelblue', alpha=0.7, label='Без ПИД')
ax3.plot(t, np.cumsum(result_opt.rewards[:T]),  'darkorange', linewidth=1.5, label='С ПИД')
ax3.set_xlabel('Кадр')
ax3.set_ylabel('Накопленная награда')
ax3.set_title('Суммарная награда агента')
ax3.legend(fontsize=8)
ax3.grid(alpha=0.3)

plt.savefig('outputs/pid_response.png', dpi=120)
plt.show()

In [ ]:
# ── 6. Тепловая карта grid search ────────────────────────────────
kp_vals = np.linspace(0.1, 2.0, 8)
kd_vals = np.linspace(0.0, 0.5, 8)
grid    = np.zeros((len(kp_vals), len(kd_vals)))

for i, kp in enumerate(kp_vals):
    for j, kd in enumerate(kd_vals):
        pid = PIDController(Kp=kp, Ki=best['Ki'], Kd=kd)
        for e in val_errors:
            pid.step(float(e))
        grid[i, j] = pid.integral_error()

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(grid, origin='lower', aspect='auto', cmap='RdYlGn_r')
ax.set_xticks(range(len(kd_vals)))
ax.set_yticks(range(len(kp_vals)))
ax.set_xticklabels([f'{v:.2f}' for v in kd_vals])
ax.set_yticklabels([f'{v:.2f}' for v in kp_vals])
ax.set_xlabel('Kd')
ax.set_ylabel('Kp')
ax.set_title(f'Grid Search: CTE (Ki={best["Ki"]:.4f})\nТёмно-зелёный = лучше')
plt.colorbar(im, ax=ax, label='CTE')
plt.tight_layout()
plt.savefig('outputs/pid_grid_search.png', dpi=120)
plt.show()

In [ ]:
# ── 7. Итоговая таблица сравнения ─────────────────────────────────
print('=' * 55)
print(f'{"Метод":<25} {"CTE":>8} {"В полосе":>10} {"Награда":>10}')
print('-' * 55)
for name, res in [
    ('Только нейросеть',    result_raw),
    ('Нейросеть + ПИД',     result_pid),
    ('Нейросеть + ПИД opt', result_opt),
]:
    print(f'{name:<25} {res.cte:>8.4f} {res.in_lane_pct:>9.1f}% {res.total_reward:>10.0f}')
print('=' * 55)